In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import re
import base64
import dash
# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "abc123"

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.getRecordCriteria({}))


# Button Style for consistent size and appearance
button_style = {
    'margin': '5px',
    'background-color': '#CCCCCC',
    'color': 'white',
    'border': 'none',
    'padding': '10px 20px',
    'border-radius': '5px',
    'cursor': 'pointer',
    'transition': 'all 0.2s ease-in-out',
    'width': '180px'  # Fixed width to prevent resizing
}


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'my-image.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Dahboard Layout
app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div([
    html.A(
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '200px'}),
        href='https://www.snhu.edu',
        target='_blank'
    )
], style={'textAlign': 'center'}),

    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    html.Center('Peter Stegall'),
    html.Div([
    html.Div([
    html.Label("Filter by Rescue Type:"),
        
    # Filter Buttons
    html.Div([
        html.Button("Water Rescue", id='btn-water', n_clicks=0, style=button_style),
        html.Button("Mountain or Wilderness Rescue", id='btn-mountain', n_clicks=0, style=button_style),
        html.Button("Disaster Rescue", id='btn-disaster', n_clicks=0, style=button_style),
        html.Button("All Rescues", id='btn-all', n_clicks=0, style=button_style),
        html.Button("No Filter / Reset", id='btn-nofilter', n_clicks=0, style=button_style),
    ], style={'margin': '10px 0'}),
    dcc.Store(id='selected-filter', data=''),# Store the selected filter type
    dcc.Store(id='reset-trigger', data=False)# Turn reset button into a momentary button
]),
    html.Hr(),
    dash_table.DataTable(
    id='datatable-id',
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
    editable=False,
    filter_action="native",
    sort_action="native",
    sort_mode="multi",
    row_selectable="single",
    row_deletable=False,
    selected_rows=[],
    page_action="native",
    page_current=0,
    page_size=5,# Adjust max rows per page
    style_table={'overflowX': 'auto'},
    style_cell={
        'textAlign': 'left',
        'minWidth': '100px', 'width': '150px', 'maxWidth': '200px',
        'whiteSpace': 'normal'
    }
),
#This sets up the dashboard so that the chart and geolocation chart are side-by-side
    html.Br(),
     html.Hr(),
     html.Div([
        html.Div(id='graph-id', style={'flex': '1', 'padding': '10px'}),
        html.Div(id='map-id', style={'flex': '1', 'padding': '10px'})
], style={'display': 'flex', 'flexDirection': 'row'})
    ])
])
#############################################
# Interaction Between Components / Controller
#############################################
# Button callback
@app.callback(
    Output('selected-filter', 'data'),
    Output('reset-trigger', 'data'),
    Input('btn-nofilter', 'n_clicks'),
    Input('btn-all', 'n_clicks'),
    Input('btn-water', 'n_clicks'),
    Input('btn-mountain', 'n_clicks'),
    Input('btn-disaster', 'n_clicks'),
    prevent_initial_call=True
)
def set_filter(n_no, n_all, n_water, n_mountain, n_disaster):
    ctx = dash.callback_context

    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    clicked_id = ctx.triggered[0]['prop_id'].split('.')[0]

    if clicked_id == 'btn-nofilter':
        return '', True
    elif clicked_id == 'btn-all':
        return 'All', False
    elif clicked_id == 'btn-water':
        return 'Water', False
    elif clicked_id == 'btn-mountain':
        return 'Mountain', False
    elif clicked_id == 'btn-disaster':
        return 'Disaster', False
# Button activity         
@app.callback(
    Output('btn-nofilter', 'style'),
    Output('btn-all', 'style'),
    Output('btn-water', 'style'),
    Output('btn-mountain', 'style'),
    Output('btn-disaster', 'style'),
    Input('selected-filter', 'data')
)
def update_button_styles(selected):
    def style(active):
        return {
            'margin': '5px',
        'backgroundColor': '#007BFF' if active else '#e0e0e0',
        'color': 'white' if active else 'black',
        'border': '2px solid #999',
        'padding': '10px 20px',
        'borderRadius': '8px',
        'cursor': 'pointer',
        'boxShadow': 'inset 2px 2px 5px rgba(0, 0, 0, 0.3)' if active else '2px 2px 5px rgba(0, 0, 0, 0.3)',
        'fontWeight': 'bold',
        'transition': 'all 0.2s ease-in-out',
        }
    return (
        style(selected == 'NoFilter'),
        style(selected == 'All'),
        style(selected == 'Water'),
        style(selected == 'Mountain'),
        style(selected == 'Disaster')
    )
# Button activated filter queries
@app.callback(
    Output('datatable-id', 'data'),
    Input('selected-filter', 'data')
)
def update_dashboard(rescue_type):
    # Define regex patterns
    labRegex = re.compile(".*lab.*", re.IGNORECASE)
    chesaRegex = re.compile(".*chesa.*", re.IGNORECASE)
    newRegex = re.compile(".*newf.*", re.IGNORECASE)
    germanRegex = re.compile(".*german.*", re.IGNORECASE)
    alaskanRegex = re.compile(".*mala.*", re.IGNORECASE)
    oldRegex = re.compile(".*old english.*", re.IGNORECASE)
    huskyRegex = re.compile(".*husk.*", re.IGNORECASE)
    rottRegex = re.compile(".*rott.*", re.IGNORECASE)
    goldenRegex = re.compile(".*golden.*", re.IGNORECASE)
    bloodRegex = re.compile(".*blood.*", re.IGNORECASE)
    doberRegex = re.compile(".*dober.*", re.IGNORECASE)

    # Water Rescue Query
    water_query = {
    "animal_type": "Dog",
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}, 
    "$or": [
         {"breed": {"$regex": newRegex}},
         {"breed": {"$regex": chesaRegex}},
         {"breed": {"$regex": labRegex}},
    ]
}

    # Mountain Rescue Query
    mountain_query = {
        "animal_type": "Dog",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
        "$or": [
            {"breed": {"$regex": germanRegex}},
            {"breed": {"$regex": alaskanRegex}},
            {"breed": {"$regex": oldRegex}},
            {"breed": {"$regex": huskyRegex}},
            {"breed": {"$regex": rottRegex}},
        ]
    }

    # Disaster Rescue Query
    disaster_query = {
        "animal_type": "Dog",
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
        "$or": [
            {"breed": {"$regex": germanRegex}},
            {"breed": {"$regex": goldenRegex}},
            {"breed": {"$regex": bloodRegex}},
            {"breed": {"$regex": doberRegex}},
            {"breed": {"$regex": rottRegex}},
        ]
    }

    # Query call on filter click
    if not rescue_type:
        df = pd.DataFrame(shelter.getRecordCriteria({}))
        return df.to_dict('records')

    elif rescue_type == 'Water':
        return pd.DataFrame(shelter.getRecordCriteria(water_query)).to_dict('records')

    elif rescue_type == 'Mountain':
        return pd.DataFrame(shelter.getRecordCriteria(mountain_query)).to_dict('records')

    elif rescue_type == 'Disaster':
        return pd.DataFrame(shelter.getRecordCriteria(disaster_query)).to_dict('records')

    elif rescue_type == 'All':
        water_data = pd.DataFrame(shelter.getRecordCriteria(water_query))
        mountain_data = pd.DataFrame(shelter.getRecordCriteria(mountain_query))
        disaster_data = pd.DataFrame(shelter.getRecordCriteria(disaster_query))

        combined = pd.concat([water_data, mountain_data, disaster_data]).drop_duplicates()
        return combined.to_dict('records')
    
    elif rescue_type == 'NoFilter':
        df = pd.DataFrame(shelter.getRecordCriteria({}))
        return df.to_dict('records')
    
# Reset selected row when 'No Filter / Reset' is clicked
@app.callback(
    Output('datatable-id', 'selected_rows'),
    Input('btn-nofilter', 'n_clicks'),
    Input('btn-all', 'n_clicks'),
    Input('btn-water', 'n_clicks'),
    Input('btn-mountain', 'n_clicks'),
    Input('btn-disaster', 'n_clicks'),
    prevent_initial_call=True
)
def reset_selected_rows(*args):
    return []
     
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return html.Div("No data to display")
    
    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' in dff.columns:
        breed_counts = dff['breed'].value_counts().reset_index()
        breed_counts.columns = ['breed', 'count']

        # Use breed_counts instead of dff
        fig = px.pie(
    breed_counts,
    names='breed',
    values='count',
    title='Breed Distribution',
    labels={'breed': 'Breed', 'count': 'Count'}
)

        fig.update_traces(textposition='inside', textinfo='percent+label')

        return [dcc.Graph(figure=fig)]
    else:
        return html.Div("Breed column missing in data")
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Check if any column is selected
    if not selected_columns:
        return []  # No columns selected, no styling applied
    
    # Apply background color to selected columns
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows"),
     Input('selected-filter', 'data') 
    ])
def update_map(viewData, index, selected_filter):
     # If reset is pressed, clear the map
    if selected_filter == 'NoFilter':
        return [html.P("Map cleared. Select a row to display location.")]
    # 1. Defensive check: No data
    if not viewData or not index:
        return [html.P("No data to display on the map.")]

    # 2. Convert input to DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # 3. Check for required columns
    required_columns = ['location_lat', 'location_long', 'breed', 'name']
    missing_columns = [col for col in required_columns if col not in dff.columns]
    if missing_columns:
        return [html.P(f"Missing required column(s): {', '.join(missing_columns)}")]

    # 4. Handle selected row (or default to 0)
    if not index or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # 5. Avoid row index error
    if row >= len(dff):
        return [html.P("Selected row is out of range.")]

    # 6. Extract location + info
    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row].get('name', '')
    if not name:
        name = "No name available"


    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(name)
                ])
            ])
        ])
    ]


app.run_server(debug=True)


Dash app running on http://127.0.0.1:13442/
